<a href="https://colab.research.google.com/github/Kaneriah43/Flyrank_Ml_Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kaneriah43/Flyrank_Ml_Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I use five pre-decision features for the March 2026 development slice:

1. `gsc_clicks_30d` — total Google Search clicks in the historical 30-day window.
2. `gsc_impressions_30d` — total Google Search impressions in the historical 30-day window.
3. `gsc_avg_position_30d` — average Google Search position in the historical window.
4. `ga4_sessions_30d` — total GA4 sessions in the historical 30-day window.
5. `content_visible_query_count` — number of queries for which the content was visible in the historical query window.

The historical performance features are calculated only from data available before
the decision moment. Count features are filled with zero when no activity is
observed. Missing search position is kept as missing rather than treating a
missing position as a real ranking value.

The final feature vector does not contain future outcome information, client or
content identifiers, private queries, URLs, or existing product decision flags.

In [2]:
%pip -q install duckdb huggingface_hub

In [4]:
import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): ")

Paste your Hugging Face READ token (hf_...): ··········


In [5]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')")

In [6]:
REL = "hf://datasets/FlyRank/internship-warehouse"

In [7]:
TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

decision_date = "2026-03-01"

features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_clicks) AS gsc_clicks_30d,
        SUM(gsc_impressions) AS gsc_impressions_30d,
        AVG(gsc_avg_position) AS gsc_avg_position_30d,
        SUM(ga4_sessions) AS ga4_sessions_30d

    FROM {TABLES['fact_daily']}

    WHERE report_date >= DATE '2026-01-31'
      AND report_date < DATE '2026-03-01'

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

query_features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        MAX(content_visible_query_count) AS content_visible_query_count
    FROM {TABLES['fact_query_90d']}
    WHERE window_end < DATE '2026-03-01'
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

features = features.merge(
    query_features,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

features["gsc_clicks_30d"] = features["gsc_clicks_30d"].fillna(0)
features["gsc_impressions_30d"] = features["gsc_impressions_30d"].fillna(0)
features["ga4_sessions_30d"] = features["ga4_sessions_30d"].fillna(0)
features["content_visible_query_count"] = (
    features["content_visible_query_count"].fillna(0)
)

features["gsc_position_missing"] = (
    features["gsc_avg_position_30d"].isna().astype(int)
)

features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_clicks_30d,gsc_impressions_30d,gsc_avg_position_30d,ga4_sessions_30d,content_visible_query_count,gsc_position_missing
0,client_3ffa76342f366962,content_82038f437a11af25,0.0,0.0,NaN,0.0,0.0,1
1,client_3ffa76342f366962,content_cd41689d5b780bed,0.0,0.0,NaN,0.0,0.0,1
2,client_3ffa76342f366962,content_b4df304c7bcdddd7,0.0,0.0,NaN,0.0,0.0,1
3,client_3ffa76342f366962,content_5980edca3b6eb7db,0.0,0.0,NaN,0.0,0.0,1
4,client_3ffa76342f366962,content_ff15cb04451fbe0b,0.0,0.0,NaN,0.0,0.0,1


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

- **`gsc_clicks_30d`** — measures organic Google Search clicks during the historical 30-day window. Missing activity is filled with 0. It is available before the prediction moment because the window ends before 2026-03-01.

- **`gsc_impressions_30d`** — measures how often the content appeared in Google Search during the historical 30-day window. Missing activity is filled with 0. It is available before the prediction moment because the window ends before 2026-03-01.

- **`gsc_avg_position_30d`** — measures the observed average Google Search position during the historical window. Missing position is kept as missing rather than interpreted as a ranking value. It is available before the prediction moment because only the historical window is used.

- **`ga4_sessions_30d`** — measures GA4 sessions during the historical 30-day window. Missing activity is filled with 0 where there is no observed activity. It is available before the prediction moment because the window ends before 2026-03-01.

- **`content_visible_query_count`** — measures how many queries were associated with visible content in the available historical query window. Missing values are filled with 0. Only windows ending before the decision moment are used, so the feature does not intentionally use future information.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I checked the feature vector for three leakage risks.

**Label-derived leakage:** the future organic-search-click outcome must not be
included as a feature. I deliberately created a `future_gsc_clicks` feature from
the outcome period to test this risk. This feature is expected to produce an
artificially strong result because it directly contains information about the
label.

**Future-window leakage:** the historical GSC and GA4 features end before the
decision moment. Query features are restricted to windows that end before the
decision moment. Therefore, the outcome window is not intentionally included
in the normal feature vector.

**Decision-derived leakage:** existing product flags, scores, or decisions are
not used as model features because they may encode a decision already made by
the system.

After the deliberate leakage test, `future_gsc_clicks` is removed. The final
feature vector contains only the five pre-decision features.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
leak = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS future_gsc_clicks
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
    GROUP BY client_hash_id, content_hash_id
""").df()

leaky_features = features.merge(
    leak,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

leaky_features["future_gsc_clicks"] = (
    leaky_features["future_gsc_clicks"].fillna(0)
)

leaky_features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_clicks_30d,gsc_impressions_30d,gsc_avg_position_30d,ga4_sessions_30d,content_visible_query_count,gsc_position_missing,future_gsc_clicks
0,client_3ffa76342f366962,content_82038f437a11af25,0.0,0.0,NaN,0.0,0.0,1,0.0
1,client_3ffa76342f366962,content_cd41689d5b780bed,0.0,0.0,NaN,0.0,0.0,1,0.0
2,client_3ffa76342f366962,content_b4df304c7bcdddd7,0.0,0.0,NaN,0.0,0.0,1,0.0
3,client_3ffa76342f366962,content_5980edca3b6eb7db,0.0,0.0,NaN,0.0,0.0,1,0.0
4,client_3ffa76342f366962,content_ff15cb04451fbe0b,0.0,0.0,NaN,0.0,0.0,1,0.0


In [10]:
leaky_features = leaky_features.drop(
    columns=["future_gsc_clicks"]
)

print(leaky_features.columns.tolist())

['client_hash_id', 'content_hash_id', 'gsc_clicks_30d', 'gsc_impressions_30d', 'gsc_avg_position_30d', 'ga4_sessions_30d', 'content_visible_query_count', 'gsc_position_missing']


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- **`future_gsc_clicks`** — excluded because it is derived from the future outcome and directly leaks the label.

- **Future GSC impressions and positions** — excluded because they are not available at the decision moment.

- **Future GA4 sessions and engagement metrics** — excluded because they describe behaviour after the decision moment.

- **Future `fact_query_90d` windows** — excluded when the window extends into or beyond the outcome period because this would introduce future information.

- **`client_hash_id`** — retained only as an identifier for joining and grouping, not as a model feature, because the model could memorize client-specific patterns.

- **`content_hash_id`** — retained only for identification and joining, not as a model feature, because it identifies an individual content item rather than providing a general predictive signal.

- **`report_date`** — not used as a raw model feature because it primarily identifies the observation time.

- **`client_has_gsc` and `client_has_ga4`** — excluded from the feature vector because they describe access/availability rather than the content's observed performance.

- **Product decision flags or existing-system scores** — excluded because they can encode an existing decision rule and create circular predictions.

- **Private queries, URLs, and client names** — excluded to avoid unnecessary exposure of private information.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.